[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_59_Web_API_FastAPI.ipynb)

# Lesson 59 — paper-distiller: Web API (FastAPI, Async Jobs & Deployment)

**Phase 6 · Lesson 4 of 10** — turning the CLI you shipped in Lesson 58 into a real, deployable HTTP service.

| Lesson | Topic | Status |
|---|---|---|
| 56 | Phase 6 Kickoff — architecture, scaffold | ✅ |
| 57 | Core pipeline — section detection, scanned-PDF fallback, batch distiller, golden eval harness | ✅ |
| 58 | Typer CLI + PyPI packaging | ✅ |
| **59** | **Web API — FastAPI service, async jobs, deployment** | 👉 today |
| 60 | OSS Growth — README, badges, contribution funnel | next |
| 61 | agent-bench — turning paper-distiller into a benchmarked agent | later |
| 62 | External data sources (Semantic Scholar, OpenReview) | later |
| 63 | Safety & abuse controls for a public API | later |
| 64 | Launch Day — Phase 6 capstone retrospective | later |

**Where you are:** `paper-distiller` already has a working pipeline (fetch → extract → code example), a section-aware extractor with scanned-PDF OCR fallback, a batch distiller, a golden eval harness, and a polished Typer CLI published (in spirit) to PyPI. Today it only runs on *your* machine, from *your* terminal. The next unlock for an open-source tool is letting other programs — a web front-end, another agent, a CI job, a Slack bot — call it over HTTP without installing anything.

## Why a Web API (and not just the CLI)?

The CLI from Lesson 58 is great for *you*, sitting at a terminal. It is the wrong interface the moment someone else — a teammate, a website, another agent — wants to use paper-distiller programmatically. That's what an HTTP API buys you:

| Need | CLI | Web API |
|---|---|---|
| Called by a human at a terminal | ✅ great | ⚠️ overkill |
| Called by a website / frontend | ❌ can't shell out from a browser | ✅ `fetch()` |
| Called by another AI agent (tool use) | ⚠️ needs subprocess + parsing | ✅ structured JSON in/out |
| Long-running batch jobs | ⚠️ blocks the terminal | ✅ background job + polling |
| Multiple concurrent users | ❌ one process, one user | ✅ many requests, one server |
| Auth / rate limiting / billing | ❌ not really a concept | ✅ API keys, quotas |

This lesson builds `paper_distiller/api.py`: a FastAPI service with a **synchronous** endpoint for single-paper digests, an **asynchronous job** pattern for batches (because batches can take minutes — you never want an HTTP client to sit on an open connection that long), simple API-key auth, a token-bucket rate limiter (the same pattern from Lesson 53's production deployment lesson, now applied to a *new* project), and a Dockerfile to ship it.

**New concept vs. Lesson 53:** back then you productionized `auto_researcher`. Today you're doing it for `paper-distiller`, but the interesting new piece is the **sync-vs-async endpoint split** — a single paper digest is fast enough to answer directly (~5-15s), but a batch of 20 papers is not, so it needs a job-queue-shaped API even though we're not using a real queue (Celery/RQ) yet — just `BackgroundTasks` and an in-memory store, with an explicit note on when you'd swap in Redis.

In [ ]:
# Setup — Colab
!pip install -q fastapi "uvicorn[standard]" httpx anthropic pydantic pdfplumber requests rich nest_asyncio python-multipart

import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import nest_asyncio
nest_asyncio.apply()  # lets us run an event loop inside Colab's already-running loop

print("Setup complete.")


## Architecture

```
                     ┌─────────────────────────────────────────┐
                     │              FastAPI app                 │
 HTTP client ──────▶ │  APIKeyAuth ──▶ RateLimiter ──▶ route     │
 (curl / browser /   │                                            │
  another agent)     │  POST /distill      (sync, one paper)     │
                     │  POST /batch        (async, many papers)  │
                     │  GET  /jobs/{id}     (poll status)         │
                     │  GET  /jobs/{id}/result (fetch when done)  │
                     │  GET  /jobs/{id}/stream (SSE progress)     │
                     │  GET  /health                              │
                     └───────────────┬───────────────────────────┘
                                      │
                     ┌────────────────▼────────────────┐
                     │   paper_distiller core (L56-57)   │
                     │   FetchLayer → ExtractLayer       │
                     └────────────────┬────────────────┘
                                      │
                                 Claude API
```

Two request shapes, two lifecycles:

- **`/distill`** is a normal request/response call. The client waits ~5-15s and gets the digest back in the same HTTP response. Simple, but only viable because one paper is fast.
- **`/batch`** returns a `job_id` in <100ms and does the real work in a `BackgroundTask`. The client polls `/jobs/{id}` (or subscribes to `/jobs/{id}/stream` for live progress) until `status == "done"`, then fetches `/jobs/{id}/result`. This is the same shape every async job API uses — Stripe webhooks, OpenAI's batch API, GitHub Actions runs — because HTTP connections aren't meant to stay open for minutes.

## Recreating the core (fresh Colab runtime)

Colab runtimes don't persist between lessons, so we recreate a trimmed version of the `paper_distiller` core from Lessons 56-57: `FetchLayer` (arXiv → text) and `ExtractLayer` (text → structured digest via Claude tool-calling). This is exactly what already lives in your local `paper_distiller/` package — here it's just inlined so the notebook is self-contained.

In [ ]:
import re, time, requests, pdfplumber, io
from typing import Optional, List
from pydantic import BaseModel, Field
import anthropic

client = anthropic.Anthropic()

# ---------- models ----------
class PaperDigest(BaseModel):
    arxiv_id: str
    title: str
    one_liner: str
    method_summary: str
    key_results: List[str]
    prerequisites: List[str]
    limitations: List[str]
    practitioner_tldr: str
    tags: List[str]

# ---------- fetch layer ----------
ARXIV_ID_RE = re.compile(r"(\d{4}\.\d{4,5})(v\d+)?")

def parse_arxiv_id(raw: str) -> str:
    m = ARXIV_ID_RE.search(raw)
    if not m:
        raise ValueError(f"Could not parse an arXiv ID out of: {raw!r}")
    return m.group(1)

def fetch_metadata(arxiv_id: str) -> dict:
    r = requests.get(f"http://export.arxiv.org/api/query?id_list={arxiv_id}", timeout=15)
    r.raise_for_status()
    title_m = re.search(r"<title>(.*?)</title>", r.text, re.S)
    title = title_m.group(1).strip() if title_m else arxiv_id
    title = re.sub(r"\s+", " ", title).replace(f"{arxiv_id}v1", "").strip()
    return {"title": title}

def fetch_pdf_text(arxiv_id: str, max_pages: int = 12) -> str:
    r = requests.get(f"https://arxiv.org/pdf/{arxiv_id}", timeout=30)
    r.raise_for_status()
    with pdfplumber.open(io.BytesIO(r.content)) as pdf:
        pages = pdf.pages[:max_pages]
        return "\n\n".join(p.extract_text() or "" for p in pages)

# ---------- extract layer ----------
EXTRACT_TOOL = {
    "name": "emit_digest",
    "description": "Emit a structured practitioner-focused digest of a research paper.",
    "input_schema": {
        "type": "object",
        "properties": {
            "one_liner": {"type": "string"},
            "method_summary": {"type": "string"},
            "key_results": {"type": "array", "items": {"type": "string"}},
            "prerequisites": {"type": "array", "items": {"type": "string"}},
            "limitations": {"type": "array", "items": {"type": "string"}},
            "practitioner_tldr": {"type": "string"},
            "tags": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["one_liner", "method_summary", "key_results", "prerequisites",
                      "limitations", "practitioner_tldr", "tags"],
    },
}

SYSTEM_EXTRACT = (
    "You distill research papers for busy practitioner engineers, not academics. "
    "Be concrete, skip hype, call out what's actually needed to use this."
)

def extract_digest(arxiv_id: str, title: str, paper_text: str, model: str) -> PaperDigest:
    resp = client.messages.create(
        model=model,
        max_tokens=1200,
        system=SYSTEM_EXTRACT,
        tools=[EXTRACT_TOOL],
        tool_choice={"type": "tool", "name": "emit_digest"},
        messages=[{"role": "user", "content": f"Title: {title}\n\nPaper text:\n{paper_text[:15000]}"}],
    )
    tool_input = next(b.input for b in resp.content if b.type == "tool_use")
    return PaperDigest(arxiv_id=arxiv_id, title=title, **tool_input)

def distill(raw_id: str, model: str = "claude-sonnet-5") -> PaperDigest:
    arxiv_id = parse_arxiv_id(raw_id)
    meta = fetch_metadata(arxiv_id)
    text = fetch_pdf_text(arxiv_id)
    return extract_digest(arxiv_id, meta["title"], text, model)

print("Core pipeline ready — distill() end-to-end callable.")


## API schemas — the contract

The single most important design decision in a web API is its **schema**: what shape goes in, what shape comes out, and what every field means. Pydantic models double as FastAPI's request validation *and* its auto-generated OpenAPI docs — one model, two jobs.

Note the split between `DistillRequest` (single paper, synchronous) and `BatchRequest`/`JobStatusResponse` (many papers, asynchronous, with a `status` state machine: `queued → running → done | failed`).

In [ ]:
from enum import Enum
from datetime import datetime, timezone
from uuid import uuid4

class DistillRequest(BaseModel):
    arxiv_id: str = Field(..., description="arXiv ID or URL, e.g. '1706.03762' or an arxiv.org link")
    model: str = Field(default="claude-sonnet-5", description="Claude model to use for extraction")

class DistillResponse(BaseModel):
    digest: PaperDigest
    elapsed_s: float
    cost_usd: float

class BatchRequest(BaseModel):
    arxiv_ids: List[str] = Field(..., min_length=1, max_length=25)
    model: str = "claude-sonnet-5"

class JobStatus(str, Enum):
    QUEUED = "queued"
    RUNNING = "running"
    DONE = "done"
    FAILED = "failed"

class JobStatusResponse(BaseModel):
    job_id: str
    status: JobStatus
    total: int
    completed: int
    failed: int
    created_at: datetime
    updated_at: datetime

class JobResultResponse(BaseModel):
    job_id: str
    status: JobStatus
    results: List[dict]   # each item: {"arxiv_id", "digest" | "error"}

print("Schemas defined.")


## Auth and rate limiting

Two dependencies, wired into every route with FastAPI's `Depends()`:

- **`require_api_key`** — reads an `X-API-Key` header, checks it against a known set (env var in production, a hardcoded demo key here), raises `401` if missing/wrong.
- **`RateLimiter`** — the same token-bucket pattern from Lesson 53, keyed by API key this time instead of by IP, so each caller gets their own bucket. `429 Too Many Requests` with a `Retry-After` header when the bucket is empty.

In [ ]:
from fastapi import Header, HTTPException, status
import threading

VALID_API_KEYS = {"demo-key-123"}  # in production: load from env / a database

def require_api_key(x_api_key: Optional[str] = Header(default=None)) -> str:
    if x_api_key is None or x_api_key not in VALID_API_KEYS:
        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED, detail="Missing or invalid X-API-Key header")
    return x_api_key


class TokenBucket:
    """Per-key token bucket. capacity tokens, refilled at refill_rate tokens/sec."""
    def __init__(self, capacity: int = 10, refill_rate: float = 0.2):
        self.capacity = capacity
        self.refill_rate = refill_rate
        self._buckets: dict[str, tuple[float, float]] = {}  # key -> (tokens, last_ts)
        self._lock = threading.Lock()

    def consume(self, key: str, cost: float = 1.0) -> tuple[bool, float]:
        with self._lock:
            now = time.time()
            tokens, last_ts = self._buckets.get(key, (self.capacity, now))
            tokens = min(self.capacity, tokens + (now - last_ts) * self.refill_rate)
            if tokens >= cost:
                tokens -= cost
                self._buckets[key] = (tokens, now)
                return True, 0.0
            self._buckets[key] = (tokens, now)
            wait_s = (cost - tokens) / self.refill_rate
            return False, wait_s

RATE_LIMITER = TokenBucket(capacity=5, refill_rate=0.1)  # 5 burst, 1 token per 10s steady state

def rate_limit(api_key: str = Depends(require_api_key)) -> str:
    ok, wait_s = RATE_LIMITER.consume(api_key)
    if not ok:
        raise HTTPException(
            status_code=status.HTTP_429_TOO_MANY_REQUESTS,
            detail=f"Rate limit exceeded, retry in {wait_s:.1f}s",
            headers={"Retry-After": str(int(wait_s) + 1)},
        )
    return api_key

from fastapi import Depends
print("Auth + rate limiter ready.")


## The job store — why not just a `dict`? (with the caveat that here, it is)

For a batch job we need somewhere to record: which papers are in the job, which are done, what the results were, and whether the whole thing succeeded. `JobStore` below is an in-memory dict guarded by a lock — perfectly fine for a single-process demo, and exactly what you'd rip out first when scaling to multiple server replicas (two web workers would each have their *own* in-memory store, so a client polling replica B would never see a job created on replica A). The fix in production is a shared store — Redis, or a `jobs` table in Postgres — with the exact same `get/create/update` interface, so the route code doesn't change, only `JobStore`'s internals.

In [ ]:
import asyncio
from dataclasses import dataclass, field

@dataclass
class JobRecord:
    job_id: str
    arxiv_ids: List[str]
    model: str
    status: JobStatus = JobStatus.QUEUED
    results: List[dict] = field(default_factory=list)
    created_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))
    updated_at: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

class JobStore:
    """In-memory job store. Swap for Redis/Postgres to run >1 replica."""
    def __init__(self):
        self._jobs: dict[str, JobRecord] = {}
        self._lock = threading.Lock()

    def create(self, arxiv_ids: List[str], model: str) -> JobRecord:
        job = JobRecord(job_id=str(uuid4()), arxiv_ids=arxiv_ids, model=model)
        with self._lock:
            self._jobs[job.job_id] = job
        return job

    def get(self, job_id: str) -> Optional[JobRecord]:
        with self._lock:
            return self._jobs.get(job_id)

    def update(self, job_id: str, **kwargs):
        with self._lock:
            job = self._jobs[job_id]
            for k, v in kwargs.items():
                setattr(job, k, v)
            job.updated_at = datetime.now(timezone.utc)

JOBS = JobStore()

async def run_batch_job(job_id: str):
    """Background task: distill every paper in the job, one at a time, updating status as it goes."""
    job = JOBS.get(job_id)
    JOBS.update(job_id, status=JobStatus.RUNNING)
    results = []
    loop = asyncio.get_event_loop()
    for arxiv_id in job.arxiv_ids:
        try:
            digest = await loop.run_in_executor(None, distill, arxiv_id, job.model)
            results.append({"arxiv_id": arxiv_id, "digest": digest.model_dump()})
        except Exception as e:
            results.append({"arxiv_id": arxiv_id, "error": str(e)})
        JOBS.update(job_id, results=list(results))  # progress visible mid-run
    failed = sum(1 for r in results if "error" in r)
    JOBS.update(job_id, status=JobStatus.FAILED if failed == len(results) else JobStatus.DONE)

print("JobStore + background runner ready.")


## Assembling the FastAPI app

`lifespan` (the modern replacement for `@app.on_event("startup")`) is where you'd warm caches or check the Anthropic key is set. Every route depends on `rate_limit` (which itself depends on `require_api_key`), so auth + rate limiting happen before any route body runs. Notice `/distill` awaits the pipeline directly (it's fast enough), while `/batch` returns immediately and hands the real work to `BackgroundTasks`.

In [ ]:
from fastapi import FastAPI, BackgroundTasks
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    if not os.environ.get("ANTHROPIC_API_KEY"):
        raise RuntimeError("ANTHROPIC_API_KEY not set — refusing to start")
    print("paper-distiller API starting up.")
    yield
    print("paper-distiller API shutting down.")

app = FastAPI(
    title="paper-distiller API",
    version="0.3.0",
    description="Turn an arXiv paper into a practitioner-focused digest, over HTTP.",
    lifespan=lifespan,
)

@app.get("/health")
def health():
    return {"status": "ok", "time": datetime.now(timezone.utc).isoformat()}

@app.post("/distill", response_model=DistillResponse)
async def distill_one(req: DistillRequest, api_key: str = Depends(rate_limit)):
    t0 = time.time()
    loop = asyncio.get_event_loop()
    try:
        digest = await loop.run_in_executor(None, distill, req.arxiv_id, req.model)
    except ValueError as e:
        raise HTTPException(status_code=422, detail=str(e))
    except requests.HTTPError as e:
        raise HTTPException(status_code=502, detail=f"arXiv fetch failed: {e}")
    elapsed = time.time() - t0
    return DistillResponse(digest=digest, elapsed_s=elapsed, cost_usd=round(elapsed * 0.002, 5))  # rough estimate

@app.post("/batch", status_code=202)
async def start_batch(req: BatchRequest, background_tasks: BackgroundTasks, api_key: str = Depends(rate_limit)):
    job = JOBS.create(req.arxiv_ids, req.model)
    background_tasks.add_task(run_batch_job, job.job_id)
    return {"job_id": job.job_id, "status": job.status, "poll_url": f"/jobs/{job.job_id}"}

@app.get("/jobs/{job_id}", response_model=JobStatusResponse)
def job_status(job_id: str, api_key: str = Depends(rate_limit)):
    job = JOBS.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="job not found")
    completed = sum(1 for r in job.results if "digest" in r)
    failed = sum(1 for r in job.results if "error" in r)
    return JobStatusResponse(
        job_id=job.job_id, status=job.status, total=len(job.arxiv_ids),
        completed=completed, failed=failed, created_at=job.created_at, updated_at=job.updated_at,
    )

@app.get("/jobs/{job_id}/result", response_model=JobResultResponse)
def job_result(job_id: str, api_key: str = Depends(rate_limit)):
    job = JOBS.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="job not found")
    if job.status not in (JobStatus.DONE, JobStatus.FAILED):
        raise HTTPException(status_code=409, detail=f"job is still {job.status}, not ready")
    return JobResultResponse(job_id=job.job_id, status=job.status, results=job.results)

print("Routes registered:", [r.path for r in app.routes])


## Testing without leaving Colab

You *can* run `uvicorn` in a background thread and hit `http://127.0.0.1:8000` with `httpx` — that's the closest simulation of production and what the demo below does for `/batch` polling. For quick request/response checks, FastAPI's `TestClient` (built on `httpx`, no real socket needed) is faster and is what you'd use in `pytest`. Both are shown so you recognize each pattern.

In [ ]:
from fastapi.testclient import TestClient

tc = TestClient(app)

# health check — no auth required
print(tc.get("/health").json())

# missing API key -> 401
r = tc.post("/distill", json={"arxiv_id": "1706.03762"})
print("no key:", r.status_code, r.json())

# wrong shape -> 422 (FastAPI validation, before our code even runs)
r = tc.post("/distill", json={}, headers={"X-API-Key": "demo-key-123"})
print("bad body:", r.status_code)

# real call
r = tc.post("/distill", json={"arxiv_id": "1706.03762"}, headers={"X-API-Key": "demo-key-123"})
print("distill:", r.status_code)
import json as _json
print(_json.dumps(r.json(), indent=2)[:800])


In [ ]:
# Live server + batch job polling (closer to how a real client behaves)
import threading, uvicorn, httpx

config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(2)  # let the server bind

BASE = "http://127.0.0.1:8000"
HEADERS = {"X-API-Key": "demo-key-123"}

resp = httpx.post(f"{BASE}/batch", json={"arxiv_ids": ["1706.03762", "1810.04805"]}, headers=HEADERS)
job_id = resp.json()["job_id"]
print("started job:", job_id, resp.json())

for _ in range(30):
    status = httpx.get(f"{BASE}/jobs/{job_id}", headers=HEADERS).json()
    print(f"  status={status['status']}  {status['completed']}/{status['total']} done")
    if status["status"] in ("done", "failed"):
        break
    time.sleep(3)

result = httpx.get(f"{BASE}/jobs/{job_id}/result", headers=HEADERS).json()
for item in result["results"]:
    if "digest" in item:
        print("-", item["digest"]["title"], "->", item["digest"]["one_liner"])
    else:
        print("-", item["arxiv_id"], "FAILED:", item["error"])

server.should_exit = True
thread.join(timeout=5)


## Streaming progress with Server-Sent Events (bonus)

Polling every 3s works but wastes requests and adds latency to "did it finish yet." **Server-Sent Events (SSE)** let the server push updates over one long-lived connection — simpler than WebSockets because it's one-directional (server → client) and plain HTTP, so it works through normal proxies/load balancers without special handling. `EventSourceResponse`-style generators are the standard FastAPI pattern: an `async def` that `yield`s `data: ...\n\n` chunks.

In [ ]:
from fastapi.responses import StreamingResponse

@app.get("/jobs/{job_id}/stream")
async def job_stream(job_id: str, api_key: str = Depends(rate_limit)):
    async def event_gen():
        last_completed = -1
        while True:
            job = JOBS.get(job_id)
            if job is None:
                yield f"event: error\ndata: job not found\n\n"
                return
            completed = sum(1 for r in job.results if "digest" in r or "error" in r)
            if completed != last_completed:
                payload = {"status": job.status, "completed": completed, "total": len(job.arxiv_ids)}
                yield f"data: {_json.dumps(payload)}\n\n"
                last_completed = completed
            if job.status in (JobStatus.DONE, JobStatus.FAILED):
                return
            await asyncio.sleep(1.0)

    return StreamingResponse(event_gen(), media_type="text/event-stream")

print("SSE route added: GET /jobs/{job_id}/stream")
print("Client usage: `EventSource` in JS, or `httpx.stream()` with iter_lines() in Python.")


## Deployment — Dockerfile + docker-compose + fly.toml

Same multi-stage pattern from Lesson 53 (`auto_researcher`'s production deployment), applied to `paper-distiller`. Two things worth calling out that are *new* here versus L53: the container needs `poppler-utils` for `pdfplumber`'s scanned-PDF fallback path (Lesson 57), and the `CMD` uses `--workers` — but remember from the JobStore discussion above, **more than 1 worker breaks in-memory job polling** until `JobStore` is backed by something shared. The Dockerfile below is written for `--workers 1` on purpose, with a comment explaining exactly when to change that.

In [ ]:
import pathlib

proj = pathlib.Path("/content/paper_distiller_deploy")
proj.mkdir(exist_ok=True)

(proj / "Dockerfile").write_text("""# --- builder ---
FROM python:3.11-slim AS builder
RUN apt-get update && apt-get install -y --no-install-recommends gcc poppler-utils && rm -rf /var/lib/apt/lists/*
WORKDIR /app
COPY pyproject.toml .
RUN pip wheel --no-cache-dir --wheel-dir /wheels .

# --- runtime ---
FROM python:3.11-slim
RUN apt-get update && apt-get install -y --no-install-recommends poppler-utils && rm -rf /var/lib/apt/lists/*
RUN useradd -u 1000 -m appuser
WORKDIR /app
COPY --from=builder /wheels /wheels
RUN pip install --no-cache-dir /wheels/*.whl && rm -rf /wheels
COPY paper_distiller/ ./paper_distiller/
USER appuser
EXPOSE 8000
HEALTHCHECK --interval=30s --timeout=5s CMD python -c "import httpx; httpx.get('http://localhost:8000/health').raise_for_status()"
# --workers 1: JobStore is in-memory (see Lesson 59). Bump workers only after
# swapping JobStore for Redis/Postgres, or job polling will hit the wrong replica.
CMD ["uvicorn", "paper_distiller.api:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "1"]
""")

(proj / "docker-compose.yml").write_text("""services:
  paper-distiller-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - ANTHROPIC_API_KEY=${ANTHROPIC_API_KEY}
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "python", "-c", "import httpx; httpx.get('http://localhost:8000/health').raise_for_status()"]
      interval: 30s
      timeout: 5s
      retries: 3
""")

(proj / "fly.toml").write_text("""app = "paper-distiller-api"
primary_region = "iad"

[build]

[env]
  PORT = "8000"

[http_service]
  internal_port = 8000
  force_https = true
  auto_stop_machines = true
  auto_start_machines = true
  min_machines_running = 0

[[http_service.checks]]
  path = "/health"
  interval = "30s"
  timeout = "5s"

[[vm]]
  memory = "512mb"
  cpu_kind = "shared"
  cpus = 1
""")

for f in proj.iterdir():
    print(f.name)


## 10 pitfalls in shipping AI APIs

| # | Pitfall | Why it bites | Fix |
|---|---|---|---|
| 1 | Sync endpoint for a slow operation | Client's HTTP connection times out (default ~30-60s in most clients/proxies) mid-batch | Split sync (fast) vs async job (slow) endpoints, as done here |
| 2 | In-memory job store with >1 replica | Client polls a job created on a different pod → 404 | Redis/Postgres-backed store before scaling workers/replicas |
| 3 | No rate limit | One buggy client script can burn your entire Anthropic budget in a loop | Token bucket per API key, `429` + `Retry-After` |
| 4 | Auth checked inside the route body | Easy to forget on a new route → silent open endpoint | `Depends()` at the route signature — impossible to skip by accident |
| 5 | Swallowing exceptions in background tasks | A crashed `BackgroundTask` fails silently; job hangs at `running` forever | Try/except around the work, always set `status=FAILED` on error |
| 6 | `CMD ["python", "app.py"]` (shell form implied) | `SIGTERM` from Docker/Kubernetes doesn't reach the process, container hard-kills after grace period | Exec-form CMD, as used here (already learned in L53, still applies) |
| 7 | No `422` distinction from `502`/`500` | Client can't tell "you sent a bad arXiv ID" from "our server broke" | Map `ValueError` → 422, upstream fetch failures → 502, unexpected → 500 |
| 8 | Batch endpoint with unbounded list size | `{"arxiv_ids": [... 10,000 ids ...]}` melts your background worker and budget | `Field(max_length=25)` — Pydantic enforces it before your code runs |
| 9 | SSE without a terminal event | Client's `EventSource` reconnects forever if the stream never explicitly ends | `return` after `DONE`/`FAILED` closes the generator cleanly |
| 10 | Testing only with `TestClient` | `TestClient` doesn't exercise real sockets, timeouts, or background-thread behavior — bugs only show up under `uvicorn` | Test both: `TestClient` for fast unit tests, a real running server for integration tests (as done above) |

Pitfall #2 and #6 are the two most likely to bite you specifically because you already know the *concept* from Lesson 53 (production deployment) — the risk here is copy-pasting that Dockerfile/CMD pattern without re-checking whether *this* app's assumptions (in-memory job store) still hold.

In [ ]:
# Verification checklist
checks = {
    "PaperDigest + core pipeline (distill()) defined": "distill" in dir(),
    "Pydantic API schemas defined": all(n in dir() for n in ["DistillRequest", "BatchRequest", "JobStatusResponse"]),
    "Auth dependency (require_api_key) defined": "require_api_key" in dir(),
    "Rate limiter (TokenBucket) defined": "TokenBucket" in dir(),
    "JobStore + background runner defined": all(n in dir() for n in ["JobStore", "run_batch_job"]),
    "FastAPI app assembled with routes": len(app.routes) >= 6,
    "SSE streaming route added": any(r.path == "/jobs/{job_id}/stream" for r in app.routes),
    "Deployment files written": (proj / "Dockerfile").exists() and (proj / "docker-compose.yml").exists() and (proj / "fly.toml").exists(),
}

for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

print()
print("Routes:")
for r in app.routes:
    if hasattr(r, "methods"):
        print(f"  {list(r.methods)} {r.path}")


## Summary

| Concept | What you built |
|---|---|
| Sync vs async endpoint design | `/distill` (await + return) vs `/batch` (`202` + `BackgroundTasks` + poll) |
| Pydantic request/response schemas | `DistillRequest/Response`, `BatchRequest`, `JobStatusResponse`/`JobResultResponse` |
| Dependency-injected auth | `require_api_key` via `Header()` + `Depends()` |
| Rate limiting | Per-key token bucket, `429` + `Retry-After` |
| In-memory job store (+ its scaling limit) | `JobStore`/`JobRecord`, explicit note on Redis/Postgres swap for >1 replica |
| Error mapping | `ValueError→422`, `HTTPError→502`, uncaught→500 |
| Server-Sent Events | `StreamingResponse` + `text/event-stream` generator with a terminal condition |
| Testing | `TestClient` for unit-style checks, real `uvicorn` thread + `httpx` for integration |
| Deployment | Multi-stage Dockerfile (with `poppler-utils` for OCR fallback), docker-compose, fly.toml |

### Homework

1. Add an `X-Request-ID` middleware (from Lesson 53's pattern) so every log line and error response can be traced to one request.
2. Swap `JobStore` for a Redis-backed version (`redis-py`, `HSET`/`HGETALL` per job) and verify two `uvicorn --workers 2` processes can both see the same job.
3. Add a `DELETE /jobs/{id}` route with a background-task cancellation flag (`asyncio.Task.cancel()`), and handle the partially-completed case in `/jobs/{id}/result`.
4. Write 3 `pytest` tests using `TestClient`: missing auth, malformed body, and a mocked `distill()` (so the test suite doesn't burn real API budget).
5. Point a browser's `EventSource` at `/jobs/{id}/stream` from a tiny static HTML page and watch progress update live — no polling JS required.

### Next: Lesson 60 — OSS Growth

Now that `paper-distiller` has a CLI (L58) *and* an API (L59), it's actually usable by someone who isn't you. Lesson 60 covers the unglamorous but essential growth work: a README that sells the project in 30 seconds, badges (CI/PyPI/coverage — you generated these for `auto_researcher` in L55, now for real on GitHub), a `CONTRIBUTING.md` funnel tuned for first-time contributors, GitHub Discussions/Issues templates, and how to actually get the first 10 stars and first outside PR.